# Imports and setup

In [16]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score, precision_score, make_scorer
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline

from churn.config import SEED, TARGET, RAW_PATH
from churn.data import load_raw, clean
from churn.features import build_preprocessor

# Load and clean

In [17]:
df = clean(load_raw(RAW_PATH))

# Train/test split

In [18]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

# Dummy baseline

In [19]:
make_scorer(precision_score, zero_division=0)
scoring = ["roc_auc", "recall", "precision"]

dummy = DummyClassifier(strategy="prior")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
res = cross_validate(dummy, X_train, y_train, cv=cv, scoring=scoring)
print({k: round(v.mean(), 3) for k, v in res.items() if k.startswith("test_")})

{'test_roc_auc': np.float64(0.5), 'test_recall': np.float64(0.0), 'test_precision': np.float64(0.0)}


d:\Practice\Data-Scientist\Churn-Prediction\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Practice\Data-Scientist\Churn-Prediction\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Practice\Data-Scientist\Churn-Prediction\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

# Logistic regression baseline

In [20]:
model = Pipeline([
    ("prep", build_preprocessor()),
    ("clf", LogisticRegression(max_iter=1000, random_state=SEED))
])
res = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
print({k: round(v.mean(), 3) for k, v in res.items() if k.startswith("test_")})

{'test_roc_auc': np.float64(0.858), 'test_recall': np.float64(0.575), 'test_precision': np.float64(0.665)}
